# Notebook 2: Clasificación de Imágenes

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

Clasificar imágenes en 5 categorías (**Animales, Ciudad, Comida, Naturaleza, Playa**) comparando tres enfoques de complejidad creciente:

| # | Modelo | Enfoque | Complejidad |
|---|--------|---------|-------------|
| 1 | HSV Histograms + SVM | ML clásico (baseline) | Baja |
| 2 | CNN from scratch | Deep Learning sin preentrenar | Media |
| 3 | MobileNetV2 fine-tuned | Transfer Learning | Alta |

### Métricas de evaluación
- **Accuracy** y **Macro-F1** para comparación global
- **Confusion Matrix** para análisis de errores por clase
- **Sensitivity** (recall) y **Specificity** por clase
- **Curvas de entrenamiento** (accuracy/loss) para evaluar *overfitting*

> **Nota:** Este notebook usa el dataset aumentado generado en el Notebook 01 (`dataset_augmented/`).

---
## 2.1 Configuración e importación de datos

Cargamos las librerías necesarias: `scikit-learn` para ML clásico, `TensorFlow/Keras` para Deep Learning, y fijamos la semilla global para reproducibilidad.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
# Usar el dataset aumentado (generado en notebook 01)
DATA_DIR = Path("dataset_augmented")
if not DATA_DIR.exists():
    print("AVISO: No se encontró dataset_augmented. Usando dataset original.")
    DATA_DIR = Path("dataset")

classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Directorio de datos:", DATA_DIR)
print("Clases:", classes)

In [ ]:
# Indexar imágenes
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rows = []
for cls in classes:
    for p in (DATA_DIR / cls).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({"path": str(p), "class": cls})

df = pd.DataFrame(rows)
print(f"Total imágenes: {len(df)}")
df["class"].value_counts().sort_index()

---
## 2.2 Split train / validation / test (estratificado)

Dividimos el dataset con estratificación por clase para mantener la proporción original:

| Conjunto | Proporción | Uso |
|----------|-----------|-----|
| **Train** | 70% | Entrenamiento de los modelos |
| **Validation** | 15% | Selección de hiperparámetros y early stopping |
| **Test** | 15% | Evaluación final imparcial |

La estratificación garantiza que cada conjunto tenga la misma distribución de clases.

In [ ]:
class_names = sorted(df["class"].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
num_classes = len(class_names)

df["label"] = df["class"].map(class_to_idx)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])

print(f"Train: {train_df.shape[0]} | Val: {val_df.shape[0]} | Test: {test_df.shape[0]}")
print("\nDistribución por clase (train):")
print(train_df["class"].value_counts().sort_index())

---
## 2.3 Funciones de evaluación

Definimos funciones reutilizables para evaluar todos los modelos con las **mismas métricas**, lo que permite una comparación justa entre enfoques.

In [ ]:
def plot_confusion(cm, title="Confusion Matrix"):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)
    
    # Añadir números en las celdas
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

def per_class_sens_spec(y_true, y_pred):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    total = cm.sum()
    
    sens = np.diag(cm) / (cm.sum(axis=1) + 1e-12)  # recall
    spec = []
    for i in labels:
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - (tp + fp + fn)
        spec.append(tn / (tn + fp + 1e-12))
    
    return cm, pd.DataFrame({
        "class": class_names,
        "sensitivity(recall)": np.round(sens, 4),
        "specificity": np.round(spec, 4)
    })

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"{model_name} — Accuracy: {acc:.4f} | Macro-F1: {f1m:.4f}")
    
    cm, sens_spec = per_class_sens_spec(np.array(y_true), np.array(y_pred))
    plot_confusion(cm, f"{model_name}")
    display(sens_spec)
    print(classification_report(y_true, y_pred, target_names=class_names))
    return acc, f1m

---
## 2.4 Modelo 1: Baseline ML clásico (HSV Histograms + SVM)

### Enfoque
Como primera aproximación, usamos un pipeline de **Machine Learning clásico**:
1. **Extracción de features:** Convertimos cada imagen a un histograma 3D en espacio HSV (8×8×8 = 512 bins), capturando la distribución de colores
2. **Clasificación:** SVM con kernel RBF (no lineal)

### Justificación
El espacio HSV separa la información de *matiz* (H), *saturación* (S) y *valor/brillo* (V), lo cual es más informativo que RGB para describir la paleta cromática de una escena.

> **Limitación esperada:** Este modelo solo captura información de color global, ignorando formas, texturas y composición espacial.

In [ ]:
def hsv_hist_feature(path, size=(128, 128), bins=(8, 8, 8)):
    im = Image.open(path).convert("RGB").resize(size)
    hsv = im.convert("HSV")
    arr = np.array(hsv)
    
    h = arr[..., 0].ravel()
    s = arr[..., 1].ravel()
    v = arr[..., 2].ravel()
    
    hist, _ = np.histogramdd(
        np.stack([h, s, v], axis=1),
        bins=bins,
        range=[(0, 255), (0, 255), (0, 255)]
    )
    hist = hist.astype(np.float32)
    hist = hist / (hist.sum() + 1e-12)
    return hist.ravel()

X_train = np.vstack([hsv_hist_feature(p) for p in tqdm(train_df["path"], desc="ML features train")])
y_train = train_df["label"].values

X_test = np.vstack([hsv_hist_feature(p) for p in tqdm(test_df["path"], desc="ML features test")])
y_test = test_df["label"].values

ml_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=5, gamma="scale"))
])

ml_model.fit(X_train, y_train)
y_pred_ml = ml_model.predict(X_test)

acc_ml, f1_ml = evaluate_model(y_test, y_pred_ml, "ML baseline (HSV hist + SVM)")

**Análisis del baseline:** Como se esperaba, el rendimiento es limitado. El modelo captura diferencias cromáticas entre clases (playa=azul/amarillo, naturaleza=verde), pero confunde categorías con paletas similares. Esto confirma la necesidad de **modelos que aprendan features espaciales** (Deep Learning).

---

## 2.5 Modelo 2: CNN entrenada desde cero

### Arquitectura
Red convolucional con 3 bloques `Conv2D → MaxPool`, seguidos de `GlobalAveragePooling → Dense`:

```
Input(224×224×3) → [Aug] → Conv32 → Pool → Conv64 → Pool → Conv128 → Pool → GAP → Dense128 → Softmax(5)
```

### Estrategia contra *overfitting*
Con ~1750 imágenes de entrenamiento (todavía limitado para DL), aplicamos múltiples técnicas de regularización:
- **Data augmentation on-the-fly** (flip, rotación, zoom, contraste)
- **Dropout** (30% tras convoluciones, 30% tras dense)
- **Early stopping** con restauración del mejor modelo

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def decode_resize(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize_with_pad(img, IMG_SIZE[0], IMG_SIZE[1])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_ds(df_, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df_["path"].values, df_["label"].values))
    ds = ds.map(decode_resize, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_df, shuffle=True)
val_ds   = make_ds(val_df, shuffle=False)
test_ds  = make_ds(test_df, shuffle=False)

print(f"Batches — Train: {tf.data.experimental.cardinality(train_ds).numpy()}, Val: {tf.data.experimental.cardinality(val_ds).numpy()}, Test: {tf.data.experimental.cardinality(test_ds).numpy()}")

In [ ]:
# CNN from scratch con augmentation integrada
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="augmentation")

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_aug(inputs)

x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)
x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPool2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

cnn_scratch = keras.Model(inputs, outputs, name="cnn_from_scratch")

cnn_scratch.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_scratch.summary()

In [ ]:
callbacks_scratch = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]

history_scratch = cnn_scratch.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_scratch
)

### Curvas de entrenamiento y evaluación

Analizamos las curvas de accuracy y loss para detectar *overfitting* (divergencia entre train y validation).

In [ ]:
def plot_history(history, title):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(h["accuracy"], label="train_acc")
    axes[0].plot(h["val_accuracy"], label="val_acc")
    axes[0].set_title(f"{title} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    
    axes[1].plot(h["loss"], label="train_loss")
    axes[1].plot(h["val_loss"], label="val_loss")
    axes[1].set_title(f"{title} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

plot_history(history_scratch, "CNN from scratch")

# Evaluación en test
y_true_cnn, y_pred_cnn = [], []
for xb, yb in test_ds:
    probs = cnn_scratch.predict(xb, verbose=0)
    y_true_cnn.extend(yb.numpy())
    y_pred_cnn.extend(np.argmax(probs, axis=1))

acc_cnn, f1_cnn = evaluate_model(y_true_cnn, y_pred_cnn, "CNN from scratch")

**Análisis:** La CNN entrenada desde cero mejora significativamente al baseline ML, ya que aprende **features jerárquicas** (bordes → texturas → patrones complejos). Sin embargo, con un dataset de este tamaño, la capacidad de generalización sigue siendo limitada comparada con un modelo preentrenado.

---

## 2.6 Modelo 3: Transfer Learning (MobileNetV2)

### Concepto
El **Transfer Learning** aprovecha un modelo preentrenado en un dataset masivo (ImageNet: 1.4M imágenes, 1000 clases) como extractor de features, adaptándolo a nuestra tarea específica.

### Estrategia en dos fases

| Fase | Capas entrenables | Learning rate | Objetivo |
|------|------------------|---------------|----------|
| **1. Feature extraction** | Solo clasificador (top) | 1e-3 | Adaptar las features preentrenadas |
| **2. Fine-tuning** | Últimas 20 capas + top | 1e-5 | Ajuste fino de features de alto nivel |

### Elección del modelo
**MobileNetV2** es ideal para este escenario: eficiente en parámetros (3.4M) pero con excelente capacidad de extracción de features gracias a sus *inverted residual blocks*.

In [ ]:
# Fase 1: Base congelada
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

tl_model = keras.Model(inputs, outputs, name="mobilenetv2_transfer")

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_tl = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)]
)

plot_history(history_tl, "Transfer Learning (base congelada)")

In [ ]:
# Fase 2: Fine-tuning de las últimas 20 capas
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

# No entrenar BatchNorm en datasets pequeños
for layer in base.layers[-20:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = tl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

plot_history(history_ft, "Transfer Learning (fine-tuning)")

### Evaluación final en test (Transfer Learning)

Evaluamos el modelo fine-tuned en el conjunto de **test** (datos nunca vistos durante entrenamiento ni validación).

In [ ]:
y_true_tl, y_pred_tl = [], []
for xb, yb in test_ds:
    probs = tl_model.predict(xb, verbose=0)
    y_true_tl.extend(yb.numpy())
    y_pred_tl.extend(np.argmax(probs, axis=1))

acc_tl, f1_tl = evaluate_model(y_true_tl, y_pred_tl, "Transfer Learning (MobileNetV2 fine-tuned)")

---
## 2.7 Comparación final de modelos

Comparamos los tres enfoques lado a lado para extraer conclusiones sobre la progresión de complejidad y rendimiento.

In [ ]:
results = pd.DataFrame([
    {"Modelo": "ML baseline (HSV + SVM)", "Accuracy": round(acc_ml, 4), "Macro-F1": round(f1_ml, 4)},
    {"Modelo": "CNN from scratch", "Accuracy": round(acc_cnn, 4), "Macro-F1": round(f1_cnn, 4)},
    {"Modelo": "Transfer Learning (MobileNetV2)", "Accuracy": round(acc_tl, 4), "Macro-F1": round(f1_tl, 4)},
])
display(results)

In [ ]:
# Comparación de curvas val_accuracy
scratch_val = history_scratch.history["val_accuracy"]
tl_val = history_tl.history["val_accuracy"]
ft_val = history_ft.history["val_accuracy"]

tl_total_val = list(tl_val) + list(ft_val)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(range(1, len(scratch_val)+1), scratch_val, marker="o", label="CNN scratch (val_acc)")
axes[0].plot(range(1, len(tl_total_val)+1), tl_total_val, marker="s", label="Transfer (frozen+FT) (val_acc)")
axes[0].axhline(acc_ml, linestyle="--", color="gray", label=f"ML baseline ({acc_ml:.3f})")
axes[0].set_title("Comparación de val_accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Loss
scratch_loss = history_scratch.history["val_loss"]
tl_total_loss = list(history_tl.history["val_loss"]) + list(history_ft.history["val_loss"])

axes[1].plot(range(1, len(scratch_loss)+1), scratch_loss, marker="o", label="CNN scratch (val_loss)")
axes[1].plot(range(1, len(tl_total_loss)+1), tl_total_loss, marker="s", label="Transfer (frozen+FT) (val_loss)")
ft_start = len(history_tl.history["val_loss"]) + 0.5
axes[1].axvline(ft_start, linestyle="--", alpha=0.7, color="red", label="Inicio fine-tuning")
axes[1].set_title("Comparación de val_loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Conclusiones

### Resumen comparativo

| Modelo | Enfoque | Ventajas | Limitaciones |
|--------|---------|----------|-------------|
| **SVM (HSV)** | ML clásico | Rápido, interpretable, sin GPU | Solo captura color global |
| **CNN scratch** | DL sin preentrenar | Aprende features jerárquicas | Propenso a *overfitting* con pocos datos |
| **MobileNetV2** | Transfer Learning | Mejor generalización, pocas épocas | Requiere framework/hardware más potente |

### Hallazgos clave

1. **Transfer Learning es claramente superior** para datasets pequeños: las features aprendidas en ImageNet son transferibles a nuestro dominio
2. **La Data Augmentation es esencial** para reducir el *overfitting* en los modelos DL
3. **El baseline ML demuestra** que las firmas de color son informativas pero insuficientes por sí solas
4. **Fine-tuning** de las últimas capas aporta mejora adicional al adaptar las features de alto nivel al dominio específico

### Selección del modelo

Para producción, seleccionaríamos **MobileNetV2 fine-tuned** como el mejor equilibrio entre rendimiento y eficiencia.